In [2]:
"""
raid_ai_15k.py
==============
Extracts 15,000 AI-generated samples from RAID .jsonl files.
Uses 'machine_text' column as the AI-generated text.

Output CSV has exactly two columns:
  answers  → machine_text (AI-generated)
  label    → always 1
"""

import json
import os
import pandas as pd

AI_FILES = [
    "arxiv_davinci.jsonl",
    "arxiv_dolly.jsonl",
    "wikipedia_chatgpt.jsonl",
    "wikipedia_cohere.jsonl",
    "peerread_llama.jsonl",
    "wikipedia_davinci.jsonl"
    # add more if you have them
]



N_SAMPLES    = 15_000
MIN_WORDS    = 20
RANDOM_STATE = 42
OUTPUT_FILE  = "raid_ai_15k.csv"

# ── The key fix: always use 'machine_text' ────────────────────────
TEXT_COL = "machine_text"

def load_ai_files(file_list):
    all_texts = []

    for filepath in file_list:
        if not os.path.exists(filepath):
            print(f"  [SKIP] Not found: {filepath}")
            continue

        rows = []
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))

        if not rows:
            print(f"  [SKIP] Empty: {filepath}")
            continue

        if TEXT_COL not in rows[0]:
            print(f"  [WARN] '{TEXT_COL}' not found in {filepath}. Keys: {list(rows[0].keys())}")
            continue

        count = 0
        for row in rows:
            text = str(row.get(TEXT_COL, '')).strip()
            if text:
                all_texts.append(text)
                count += 1

        print(f"  [OK] {filepath}: {count} rows")

    return all_texts


if __name__ == '__main__':

    for f in AI_FILES:
        if os.path.exists(f):
            print(f"\nPeeking: {f}")
            with open(f, 'r') as fh:
                row = json.loads(fh.readline())
                print(f"  Keys: {list(row.keys())}")
                print(f"  machine_text sample: {str(row.get('machine_text',''))[:150]}")
            break

    print("\n" + "="*50)
    print("Loading machine_text from all files...")
    print("="*50)

    texts = load_ai_files(AI_FILES)
    print(f"\nTotal raw texts: {len(texts)}")

    df = pd.DataFrame({'answers': texts})

    df['answers'] = df['answers'].str.strip().str.lower()
    df = df.drop_duplicates(subset='answers')
    before = len(df)
    df = df[df['answers'].str.split().str.len() >= MIN_WORDS]
    print(f"After dedup + short-drop: {len(df)} rows  (removed {before - len(df)})")

    if len(df) < N_SAMPLES:
        print(f"WARNING: Only {len(df)} available. Using all.")
    else:
        df = df.sample(n=N_SAMPLES, random_state=RANDOM_STATE)
        print(f"Sampled exactly {N_SAMPLES} rows.")

    df['label'] = 1
    df = df[['answers', 'label']].reset_index(drop=True)

    print("\n" + "="*50)
    print(f"Shape       : {df.shape}")
    print(f"Label counts: {df['label'].value_counts().to_dict()}")
    print(f"\nSample rows:")
    print(df[['answers','label']].head(3).to_string())

    df.to_csv(OUTPUT_FILE, index=False)
    print(f"\nSaved: {OUTPUT_FILE}")


Peeking: arxiv_davinci.jsonl
  Keys: ['prompt', 'human_text', 'machine_text', 'model', 'source', 'source_ID']
  machine_text sample: 
This paper studies the connection between ordinary Schroedinger quantum mechanics and a non-standard representation known as polymer quantum mechanic

Loading machine_text from all files...
  [OK] arxiv_davinci.jsonl: 3000 rows
  [OK] arxiv_dolly.jsonl: 3000 rows
  [OK] wikipedia_chatgpt.jsonl: 2995 rows
  [OK] wikipedia_cohere.jsonl: 2336 rows
  [OK] peerread_llama.jsonl: 586 rows
  [OK] wikipedia_davinci.jsonl: 3000 rows

Total raw texts: 14917
After dedup + short-drop: 14914 rows  (removed 2)

Shape       : (14914, 2)
Label counts: {1: 14914}

Sample rows:
                                                                                                                                                                                                                                                                                                           

In [6]:
# ── Extract human_text from same files ──────────────────────────

HUMAN_COL   = "human_text"
HUMAN_FILE  = "raid_human_15k.csv"

def load_human_files(file_list):
    all_texts = []

    for filepath in file_list:
        if not os.path.exists(filepath):
            print(f"  [SKIP] Not found: {filepath}")
            continue

        rows = []
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        rows.append(json.loads(line))
                    except json.JSONDecodeError:
                        continue   # skip blank/corrupt lines

        if not rows:
            print(f"  [SKIP] Empty: {filepath}")
            continue

        if HUMAN_COL not in rows[0]:
            print(f"  [WARN] '{HUMAN_COL}' not in {filepath}. Keys: {list(rows[0].keys())}")
            continue

        count = 0
        for row in rows:
            text = str(row.get(HUMAN_COL, '')).strip()
            if text:
                all_texts.append(text)
                count += 1

        print(f"  [OK] {filepath}: {count} human rows")

    return all_texts

print("\n" + "="*50)
print("Extracting human_text from same files...")
print("="*50)

human_texts = load_human_files(AI_FILES)
print(f"\nTotal raw human texts: {len(human_texts)}")

df_human = pd.DataFrame({'answers': human_texts})
df_human['answers'] = df_human['answers'].str.strip().str.lower()
df_human = df_human.drop_duplicates(subset='answers')
before = len(df_human)
df_human = df_human[df_human['answers'].str.split().str.len() >= MIN_WORDS]
print(f"After dedup + short-drop: {len(df_human)} rows  (removed {before - len(df_human)})")

if len(df_human) < N_SAMPLES:
    print(f"WARNING: Only {len(df_human)} available. Using all.")
else:
    df_human = df_human.sample(n=N_SAMPLES, random_state=RANDOM_STATE)
    print(f"Sampled exactly {N_SAMPLES} rows.")

df_human['label'] = 0
df_human = df_human[['answers', 'label']].reset_index(drop=True)

print(f"Shape       : {df_human.shape}")
print(f"Label counts: {df_human['label'].value_counts().to_dict()}")
print(f"\nSample rows:")
print(df_human.head(3).to_string())

df_human.to_csv(HUMAN_FILE, index=False)
print(f"\nSaved: {HUMAN_FILE}")


Extracting human_text from same files...
  [OK] arxiv_davinci.jsonl: 3000 human rows
  [OK] arxiv_dolly.jsonl: 3000 human rows
  [OK] wikipedia_chatgpt.jsonl: 2995 human rows
  [OK] wikipedia_cohere.jsonl: 2336 human rows
  [OK] peerread_llama.jsonl: 586 human rows
  [OK] wikipedia_davinci.jsonl: 3000 human rows

Total raw human texts: 14917
After dedup + short-drop: 6584 rows  (removed 0)
Shape       : (6584, 2)
Label counts: {0: 6584}

Sample rows:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [4]:
df=pd.read_csv("raid_ai_15k.csv")
df.head(6)


,answers,label
0,this paper studies the connection between ordi...,1
1,this work presents an analysis of the c2d spit...,1
2,the spectroscopic observations of ip ex hydrae...,1
3,this paper presents two classes of methods for...,1
4,thechromosphere of the sun is enigmatic and ha...,1
5,we explore the capability of detecting non-sta...,1


In [8]:
df_hum=pd.read_csv("raid_human_15k.csv")
df_hum.head(6)

,answers,label
0,a rather non-standard quantum representation o...,0
1,we discuss the results from the combined irac ...,0
2,results from spectroscopic observations of the...,0
3,we present lie group integrators for nonlinear...,0
4,"the very nature of the solar chromosphere, its...",0
5,we analyze the possibility of probing non-stan...,0
